<div style="background-color: #1e1e1e; padding: 20px; border-radius: 5px;">
    <h1 style="color: #ffffff; font-size: 40px; font-weight: 300; letter-spacing: 3px;">
        Creates Folder By the Name of Project , resets Chrome.
    </h1>
</div>

In [1]:
import pandas as pd


# project_name = 'ashiana ekansh'
# project_name = 'Akira Havenwoods'
project_name = 'DREAM COLONY'


import os
os.makedirs(project_name,exist_ok=True)


# issue handling 

# brew uninstall chromedriver
# killall -9 "Google Chrome" > /dev/null 2>&1   
# rm -rf ~/.wdm
# rm -rf /tmp/selenium_chrome_profile



#  Searches RERA, extracts subprojects.

In [2]:
# subproject iterator to save the links !!!
import os
import shutil
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

options = webdriver.ChromeOptions()
# Optional: Add options to make the window stable
options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)
wait = WebDriverWait(driver, 10)

try:
    # 1. Navigate to the RERA Rajasthan Home Page
    driver.get("https://rera.rajasthan.gov.in/Home")

    # 2. Locate the input box and search
    search_input = wait.until(
        EC.presence_of_element_located((By.NAME, "projectName"))
    )
    search_input.clear()
    search_input.send_keys(project_name)
    time.sleep(2)

    try:
        search_button = wait.until(
            EC.presence_of_element_located((By.ID, "btnSumbit"))
        )
        driver.execute_script("arguments[0].click();", search_button)
        print("Clicked Search button using ID!")

        try:
            print("Extracting table data and fetching dynamic URLs...")

            # Wait for the table rows to populate
            wait.until(
                EC.presence_of_element_located(
                    (
                        By.CSS_SELECTOR,
                        "table.table-bordered.table-striped tbody tr",
                    )
                )
            )
            table = driver.find_element(
                By.CSS_SELECTOR, "table.table-bordered.table-striped"
            )
            time.sleep(3)

            # Extract Headers
            headers = []
            header_elements = table.find_elements(By.XPATH, ".//thead//th")
            for th in header_elements:
                headers.append(th.text.strip())

            headers.append("Project_URL")  # Our custom column
            table_data = []

            rows = driver.find_elements(
                By.CSS_SELECTOR, "table.table-bordered.table-striped tbody tr"
            )
            row_count = len(rows)

            for i in range(row_count):
                current_rows = driver.find_elements(
                    By.CSS_SELECTOR,
                    "table.table-bordered.table-striped tbody tr",
                )

                if i >= len(current_rows):
                    break

                row = current_rows[i]
                cells = row.find_elements(By.XPATH, ".//td")

                expected_columns = len(headers) - 1
                if len(cells) != expected_columns:
                    continue

                row_text_data = [cell.text.strip() for cell in cells]
                project_url = None

                try:
                    buttons = row.find_elements(By.TAG_NAME, "button")

                    if not buttons:
                        print(f"No button found in row {i+1}, moving next...")
                    else:
                        view_button = buttons[0]
                        table_page_url = driver.current_url

                        # Click the view button to reveal dynamic URL
                        driver.execute_script(
                            "arguments[0].click();", view_button
                        )
                        wait.until(EC.url_changes(table_page_url))
                        time.sleep(1)

                        project_url = driver.current_url

                        # Go back to search results
                        driver.back()
                        wait.until(
                            EC.presence_of_element_located(
                                (
                                    By.CSS_SELECTOR,
                                    "table.table-bordered.table-striped tbody tr",
                                )
                            )
                        )
                        time.sleep(1)

                except Exception as e:
                    print(f"Could not fetch URL for row {i+1}: {e}")
                    try:
                        if driver.current_url != table_page_url:
                            driver.back()
                            time.sleep(2)
                    except:
                        pass

                row_text_data.append(project_url)
                table_data.append(row_text_data)
                print(f"Processed row {i+1}/{row_count}...")

            # Convert to DataFrame
            df = pd.DataFrame(table_data, columns=headers)
            print("\n--- Extracted DataFrame ---")
            print(df.head())

            # index=False prevents an extra unnamed column from being added
            df.to_csv(f"{project_name}.csv", index=False)

        except Exception as e:
            print(f"Table extraction failed: {e}")

    except Exception as e:
        print(f"Button click failed: {e}")

    time.sleep(2)
except Exception as e:
    print(f"An error occurred during automation: {e}")

finally:
    driver.quit()
    # Move the file safely into your folder structure
    if os.path.exists(f"{project_name}.csv"):
        shutil.copy(
            f"{project_name}.csv", f"{project_name}/{project_name}.csv"
        )
        os.remove(f"{project_name}.csv")
        print("Master file successfully created and relocated.")

Clicked Search button using ID!
Extracting table data and fetching dynamic URLs...
Processed row 1/1...

--- Extracted DataFrame ---
  District Name  Project Name Project Type     Promoter Name  \
0         Ajmer  DREAM COLONY      PLOTTED  Rajesh Basantani   

            Application No.              Registration No. Action  \
0  RAJ-RERA-APP-P-2023-9576  RAJ/P/2023/2878 (19/12/2023)   View   

                                         Project_URL  
0  https://rera.rajasthan.gov.in/ProjectDetail?id...  
Master file successfully created and relocated.


# SUB PROJECT ITERATION

In [3]:
df = pd.read_csv(f"{project_name}/{project_name}.csv")
df

,District Name,Project Name,Project Type,Promoter Name,Application No.,Registration No.,Action,Project_URL
0,Ajmer,DREAM COLONY,PLOTTED,Rajesh Basantani,RAJ-RERA-APP-P-2023-9576,RAJ/P/2023/2878 (19/12/2023),View,https://rera.rajasthan.gov.in/ProjectDetail?id...


# Navigates to RERA projects, captures QPR form links.

In [4]:
import os
import shutil
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import pandas as pd

print("🧹 Setting up a clean environment...")

# Create a disposable Chrome profile so we don't hit "locked data" errors
temp_profile = "/tmp/selenium_chrome_profile"
if os.path.exists(temp_profile):
    shutil.rmtree(temp_profile)

options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox') 
options.add_argument('--disable-dev-shm-usage')
options.add_argument(f'--user-data-dir={temp_profile}') # The secret weapon

print("⬇️ Fetching the correct driver...")
clean_driver_path = ChromeDriverManager().install()

print("🚀 Booting Chrome...")
driver = webdriver.Chrome(service=Service(clean_driver_path), options=options)
wait = WebDriverWait(driver, 15) # Bumped up the general wait time to 15 seconds just in case

print("✅ SUCCESS! Chrome is back online!")

print("\n--- Starting URL Navigation Phase ---")

os.makedirs(f"{project_name}/sub/", exist_ok=True)

# (Assuming 'df' is loaded from your previous chunk)
for i in range(len(df)):
    row = df.iloc[i]
    link_to_visit = row['Project_URL']
    sub_project_name = row['Project Name']
    
    if pd.isna(link_to_visit) or not link_to_visit:
        continue
        
    print(f"\nRow {i+1}: Opening {link_to_visit}")
    
    try:
        driver.get(link_to_visit)
        main_window = driver.current_window_handle
        
        # Give the entire Angular framework a heavy pause to initialize and fetch the main data
        print("Waiting for page APIs to settle...")
        time.sleep(5) 
        
        # 1. Click 'other applications'
        other_apps = wait.until(EC.presence_of_element_located((By.XPATH, "//*[contains(translate(text(), 'OTHER APPLICATIONS', 'other applications'), 'other applications')]")))
        driver.execute_script("arguments[0].click();", other_apps)
        time.sleep(2) # Pause for the tab to switch

        print("Looking for the QPR dropdown...")

        # 2. Click QPR
        qpr_summary = wait.until(EC.visibility_of_element_located((By.XPATH, "//summary[contains(., 'QPR')]")))
        driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", qpr_summary)
        time.sleep(1) 
        
        try:
            qpr_summary.click()
        except:
            driver.execute_script("arguments[0].click();", qpr_summary)
            
        # Give the table data a generous moment to fetch from the server and render
        print("Waiting for QPR table rows to populate...")
        time.sleep(4) 

        # 3. Extract the table headers
        qpr_table = wait.until(EC.visibility_of_element_located(
            (By.XPATH, "//summary[contains(., 'QPR')]/following-sibling::div[contains(@class, 'table-responsive')]//table")
        ))

        qpr_headers = [th.text.strip() for th in qpr_table.find_elements(By.XPATH, ".//thead//th")]
        qpr_headers.append("view_link") 
        
        # Wait specifically for the TR tags to exist before we count them
        try:
            wait.until(EC.presence_of_element_located((By.XPATH, "//summary[contains(., 'QPR')]/following-sibling::div[contains(@class, 'table-responsive')]//table//tbody//tr")))
        except Exception as e:
            print("No rows appeared after waiting. It might genuinely be empty.")
            
        # Find out how many rows we need to click through
        initial_qpr_rows = qpr_table.find_elements(By.XPATH, ".//tbody//tr")
        num_qpr_rows = len(initial_qpr_rows)
        
        qpr_data = []
        
        # Loop through rows by index (j)
        for j in range(num_qpr_rows):
            current_qpr_table = wait.until(EC.visibility_of_element_located(
                (By.XPATH, "//summary[contains(., 'QPR')]/following-sibling::div[contains(@class, 'table-responsive')]//table")
            ))
            current_rows = current_qpr_table.find_elements(By.XPATH, ".//tbody//tr")
            
            if j >= len(current_rows):
                break 
                
            row = current_rows[j]
            cells = row.find_elements(By.XPATH, ".//td")
            
            if not cells:
                continue
                
            row_text = [cell.text.strip() for cell in cells]
            view_link = None
            
            try:
                view_btn = row.find_element(By.TAG_NAME, "a")
                
                driver.execute_script("arguments[0].click();", view_btn)
                time.sleep(3) # Give the new page generous time to render
                
                # SCENARIO A: It opened in a New Tab
                if len(driver.window_handles) > 1:
                    for handle in driver.window_handles:
                        if handle != main_window:
                            driver.switch_to.window(handle)
                            break
                            
                    time.sleep(2) # Let the URL stabilize
                    view_link = driver.current_url
                    
                    driver.close()
                    driver.switch_to.window(main_window)
                    
                # SCENARIO B: It opened in the Same Tab
                else:
                    view_link = driver.current_url
                    driver.back()
                    print("Navigated back. Waiting for UI state to recover...")
                    time.sleep(4) # State recovery takes time!
                    
                    other_apps = wait.until(EC.presence_of_element_located((By.XPATH, "//*[contains(translate(text(), 'OTHER APPLICATIONS', 'other applications'), 'other applications')]")))
                    driver.execute_script("arguments[0].click();", other_apps)
                    time.sleep(2) 

                    qpr_summary = wait.until(EC.visibility_of_element_located((By.XPATH, "//summary[contains(., 'QPR')]")))
                    driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", qpr_summary)
                    time.sleep(1)
                    
                    try:
                        qpr_summary.click()
                    except:
                        driver.execute_script("arguments[0].click();", qpr_summary)
                    time.sleep(3) # Wait for the table to drop down and populate again
                    
            except Exception as e:
                print(f"    -> Could not fetch View link for QPR row {j+1}: {e}")
                if len(driver.window_handles) > 1:
                    driver.close()
                    driver.switch_to.window(main_window)
                    
            row_text.append(view_link)
            qpr_data.append(row_text)
            print(f"    -> Grabbed QPR row {j+1}/{num_qpr_rows}")

        # 4. Save to Pandas and CSV
        if qpr_data:
            df_qpr = pd.DataFrame(qpr_data, columns=qpr_headers)

            safe_name = str(sub_project_name).replace("/", "-").replace("\\", "-")
            csv_filename = f"{project_name}/sub/{safe_name}.csv"
            
            df_qpr.to_csv(csv_filename, index=False)
            print(f" -> Successfully saved QPR table with links to {csv_filename}")
        else:
            print(" -> Table was genuinely empty.")
        time.sleep(2)

    except Exception as e:
        print(f" -> Could not complete workflow for this link: {e}")

print("\nFinished processing all links!")
driver.quit()

🧹 Setting up a clean environment...
⬇️ Fetching the correct driver...
🚀 Booting Chrome...
✅ SUCCESS! Chrome is back online!

--- Starting URL Navigation Phase ---

Row 1: Opening https://rera.rajasthan.gov.in/ProjectDetail?id=JKWr0ciFAsqLFLgQiWRPjg
Waiting for page APIs to settle...
Looking for the QPR dropdown...
Waiting for QPR table rows to populate...
    -> Grabbed QPR row 1/9
    -> Grabbed QPR row 2/9
    -> Grabbed QPR row 3/9
    -> Grabbed QPR row 4/9
    -> Grabbed QPR row 5/9
    -> Grabbed QPR row 6/9
    -> Grabbed QPR row 7/9
    -> Grabbed QPR row 8/9
    -> Grabbed QPR row 9/9
 -> Successfully saved QPR table with links to DREAM COLONY/sub/DREAM COLONY.csv

Finished processing all links!


# Extracts and saves data. (R1, R2, R3 and Allocated details)

In [8]:
import os
import shutil
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException # <-- CRITICAL FIX IMPORT
import pandas as pd
from pathlib import Path

print("🧹 Setting up a clean environment...")

# Create a disposable Chrome profile so we don't hit "locked data" errors
temp_profile = "/tmp/selenium_chrome_profile"
if os.path.exists(temp_profile):
    shutil.rmtree(temp_profile)

options = webdriver.ChromeOptions()
options.add_argument('--no-sandbox') 
options.add_argument('--disable-dev-shm-usage')
options.add_argument(f'--user-data-dir={temp_profile}')

print("⬇️ Fetching the correct driver...")
clean_driver_path = ChromeDriverManager().install()

print("🚀 Booting Chrome...")
driver = webdriver.Chrome(service=Service(clean_driver_path), options=options)
wait = WebDriverWait(driver, 10)

print("\n--- Starting Data Extraction Phase ---")

# (Assuming project_name is defined in your environment)
# project_name = "DREAM COLONY" 

for i in Path(f"{project_name}/sub/").iterdir():
    if i.is_dir() or i.name == ".DS_Store":
        continue
    
    print(f"\n📁 Processing Sub-Project: {i.stem}")
    sub_project_name = i.stem
    sub_project_df = pd.read_csv(i)

    for index in range(len(sub_project_df)):
        row = sub_project_df.iloc[index]
        page_to_visit = row['view_link']
        quarter = row['Revised On']

        quarter_path = f"{project_name}/sub/{sub_project_name}/{quarter}"
        os.makedirs(quarter_path, exist_ok=True)
        
        print(f"\n  -> Opening Quarter: {quarter}")
        driver.get(page_to_visit)
        
        # Explicit wait for the page to stabilize
        try:
            wait.until(EC.presence_of_all_elements_located((By.XPATH, "//tr")))
        except:
            print("  -> Warning: No tables found on page or page took too long to load.")
            
        time.sleep(4) # Increased buffer time to let Angular finish binding data

        # ==========================================
        # PDF LINKS EXTRACTION
        # ==========================================
        r1_pdf_link = "Not Found"
        r2_pdf_link = "Not Found"
        r3_pdf_link = "Not Found"
        retention_pdf_link = "Not Found"
        
        try: r1_pdf_link = driver.find_element(By.XPATH, "//a[contains(@href, '/R1/')]").get_attribute("href")
        except: pass
        try: r2_pdf_link = driver.find_element(By.XPATH, "//a[contains(@href, '/R2/')]").get_attribute("href")
        except: pass
        try: r3_pdf_link = driver.find_element(By.XPATH, "//a[contains(@href, '/R3/')]").get_attribute("href")
        except: pass
        try: retention_pdf_link = driver.find_element(By.XPATH, "//a[contains(@href, '/RERAretentionAccount/')]").get_attribute("href")
        except: pass

        # ==========================================
        # UNIFIED QPR EXTRACTION (R1, R2, R3)
        # ==========================================
        print("  -> Extracting QPR Forms (R1, R2, R3) via unified parser...")
        
        row_xpath = "//div[contains(@class, 'panel-body')]//tr | //table//tr"
        all_rows = driver.find_elements(By.XPATH, row_xpath)
        total_rows = len(all_rows)

        # State Variables
        current_section = None
        current_table = None
        current_building = "N/A"
        is_r3_bottom = False

        # Data Stores
        r1_a, r1_b = [], []
        r2_a, r2_b = [], []
        r3_top, r3_bottom = [], []

        # Loop by index to allow re-fetching if elements go stale
        for row_idx in range(total_rows):
            attempts = 0
            while attempts < 3:
                try:
                    row = all_rows[row_idx]
                    row_text = row.text.strip().upper()

                    # 1. Update Section State based on headers
                    if "PART 1: ARCHITECTURAL PROGRESS" in row_text:
                        current_section = "R1"; break
                    if "PART 2: ENGINEERING PROGRESS" in row_text:
                        current_section = "R2"; break
                    if "PART 3: FINANCIAL PROGRESS" in row_text:
                        current_section = "R3"; break
                    if "PART 4:" in row_text or "V. PROJECT COMMON AREA" in row_text or "STAGE WISE TIME" in row_text:
                        current_section = None; break

                    if not current_section:
                        break # Skip this row, we aren't in a target section

                    # 2. Update Table & Building State
                    if "TABLE - A" in row_text and "SUMMARY" not in row_text:
                        current_table = "A"; break
                    if "TABLE - B" in row_text and "SUMMARY" not in row_text:
                        current_table = "B"; break
                    if "BUILDING NAME:" in row_text:
                        current_building = row.text.split(":")[-1].strip(); break

                    # 3. Extract Cell Data
                    tds = row.find_elements(By.XPATH, "./td")
                    ths = row.find_elements(By.XPATH, "./th")

                    if len(tds) >= 4 and len(ths) == 0:
                        td_texts = []
                        for td in tds:
                            inputs = td.find_elements(By.TAG_NAME, "input")
                            if inputs:
                                val = inputs[0].get_attribute("value")
                                if not val:
                                    val = driver.execute_script("return arguments[0].value;", inputs[0])
                                td_texts.append(val.strip() if val else td.text.strip())
                            else:
                                td_texts.append(td.text.strip())

                        if not any(td_texts) or "ESTIMATED AMOUNT" in str(td_texts).upper() or "INCURRED AMOUNT" in str(td_texts).upper():
                            break

                        # 4. Route Data Based on Section
                        if current_section == "R1":
                            row_data = {
                                "Building_Name": current_building,
                                "S.No.": td_texts[0],
                                "Task/Facility": td_texts[1],
                                "Percentage (Last Quarter)": td_texts[-2] if len(td_texts) >= 4 else "",
                                "Percentage (Current Quarter)": td_texts[-1] if len(td_texts) >= 4 else "",
                                "Details / Proposed": td_texts[2] if len(td_texts) >= 6 else "",
                                "docs": r1_pdf_link
                            }
                            if current_table == "A": r1_a.append(row_data)
                            elif current_table == "B": r1_b.append(row_data)

                        elif current_section == "R2":
                            row_data = {
                                "Building_Name": current_building if current_table == "A" else "N/A",
                                "S.No.": td_texts[0],
                                "Particulars": td_texts[1],
                                "Unit (₹/%)": td_texts[2] if len(td_texts) >= 5 else "",
                                "By the end of Last Quarter": td_texts[-2] if len(td_texts) >= 4 else "",
                                "By the end of Current Quarter": td_texts[-1] if len(td_texts) >= 4 else "",
                                "docs": r2_pdf_link
                            }
                            if current_table == "A": r2_a.append(row_data)
                            else: r2_b.append(row_data)

                        elif current_section == "R3":
                            # ── R3 TOP section: rows have 5 cols ──────────────────────
                            # S.No. | Particulars | Unit | Estimated Amt | Last Q | Curr Q
                            # The top section (S.No. = A/B/C) has 5+ tds.
                            #
                            # ── R3 BOTTOM section: rows 2–9 have 4 cols ──────────────
                            # S.No. | Particulars | Unit(%) | In Last Q | In Curr Q
                            # The header row for the bottom section contains
                            # "IN LAST QUARTER" / "IN CURRENT QUARTER" text and
                            # signals the switch to the bottom section.
                            # S.No. in bottom rows can be a digit OR the cell may be
                            # blank when the row spans (making td_texts[0] != "2").
                            # We detect the boundary by the header row text OR by
                            # checking that the S.No. is a pure digit (rows 2-9).

                            # Detect transition to bottom section via header row
                            joined = " ".join(td_texts).upper()
                            if "IN LAST QUARTER" in joined or "IN CURRENT QUARTER" in joined:
                                is_r3_bottom = True
                                break  # Skip this header row itself

                            # Also flip flag when we first see a pure-digit S.No.
                            # after having already collected top rows
                            sno = td_texts[0].strip()
                            if not is_r3_bottom and r3_top and sno.isdigit():
                                is_r3_bottom = True

                            if not is_r3_bottom:
                                # TOP section (5 columns):
                                # S.No. | Particulars | Unit | Estimated Amt | Last Q | Curr Q
                                row_data = {
                                    "S.No.": td_texts[0],
                                    "Particulars": td_texts[1] if len(td_texts) >= 5 else "",
                                    "Unit": td_texts[2] if len(td_texts) >= 5 else "",
                                    "Estimated Amount": td_texts[3] if len(td_texts) >= 6 else (td_texts[2] if len(td_texts) == 5 else ""),
                                    "Last Quarter (Incurred)": td_texts[-2] if len(td_texts) >= 4 else "",
                                    "Current Quarter (Incurred)": td_texts[-1] if len(td_texts) >= 4 else "",
                                    "docs": r3_pdf_link,
                                    "docs_retention": retention_pdf_link
                                }
                                r3_top.append(row_data)
                            else:
                                # BOTTOM section (4 or 5 columns):
                                # S.No. | Particulars | Unit(%) | In Last Q | In Curr Q
                                # When S.No. cell is blank (colspan row), td_texts[0] may be empty
                                row_data = {
                                    "S.No.": td_texts[0] if td_texts[0].strip() else "",
                                    "Particulars": td_texts[1] if len(td_texts) >= 4 else "",
                                    "Unit": td_texts[2] if len(td_texts) >= 5 else "",
                                    "In Last Quarter": td_texts[-2] if len(td_texts) >= 4 else "",
                                    "In Current Quarter": td_texts[-1] if len(td_texts) >= 4 else "",
                                    "docs": r3_pdf_link,
                                    "docs_retention": retention_pdf_link
                                }
                                r3_bottom.append(row_data)
                    
                    break # Success! Break out of the retry loop and move to the next row

                except StaleElementReferenceException:
                    # If Angular refreshes the DOM mid-read, re-fetch all rows and try again
                    all_rows = driver.find_elements(By.XPATH, row_xpath)
                    attempts += 1
                    time.sleep(1)
                except Exception as e:
                    break

        # ==========================================
        # SAVE QPR EXCEL FILES
        # ==========================================
        if r1_a or r1_b:
            with pd.ExcelWriter(f"{quarter_path}/Form_R1.xlsx") as writer:
                if r1_a: pd.DataFrame(r1_a).to_excel(writer, sheet_name='Table A', index=False)
                if r1_b: pd.DataFrame(r1_b).to_excel(writer, sheet_name='Table B', index=False)
            print("    ✅ Saved Form_R1.xlsx")
        else: print("    -> No data found for Form R1.")

        if r2_a or r2_b:
            with pd.ExcelWriter(f"{quarter_path}/Form_R2.xlsx") as writer:
                if r2_a: pd.DataFrame(r2_a).to_excel(writer, sheet_name='Table A', index=False)
                if r2_b: pd.DataFrame(r2_b).to_excel(writer, sheet_name='Table B', index=False)
            print("    ✅ Saved Form_R2.xlsx")
        else: print("    -> No data found for Form R2.")

        if r3_top or r3_bottom:
            with pd.ExcelWriter(f"{quarter_path}/Form_R3.xlsx") as writer:
                if r3_top: pd.DataFrame(r3_top).to_excel(writer, sheet_name='Top Table', index=False)
                if r3_bottom: pd.DataFrame(r3_bottom).to_excel(writer, sheet_name='Bottom Table', index=False)
            print("    ✅ Saved Form_R3.xlsx")
        else: print("    -> No data found for Form R3.")

        # ==========================================
        # ALLOTTEE DETAILS EXTRACTION (Saved as CSV)
        # ==========================================
        allottee_data = []
        try:
            allottee_header = driver.find_element(By.XPATH, "//h3[contains(translate(text(), 'allottee details', 'ALLOTTEE DETAILS'), 'ALLOTTEE DETAILS')]")
            allottee_container = allottee_header.find_element(By.XPATH, "./following-sibling::div[contains(@class, 'table-bordered')][1]")
            allottee_tables = allottee_container.find_elements(By.XPATH, ".//table")
            
            for tbl in allottee_tables:
                try:
                    building_info_node = tbl.find_element(By.XPATH, "./preceding-sibling::div[contains(., 'Building :')]")
                    building_text = building_info_node.text.split('\n')[0].strip()
                except:
                    building_text = "Unknown Building"
                
                rows = tbl.find_elements(By.XPATH, ".//tbody//tr")
                for row in rows:
                    tds = row.find_elements(By.XPATH, "./td")
                    if len(tds) >= 4:
                        td_texts = [td.text.strip() for td in tds]
                        allottee_data.append({
                            "Building": building_text,
                            "Sr.No.": td_texts[0],
                            "Unit/Flat Detail": td_texts[1],
                            "Booking Status": td_texts[2],
                            "Allottee Name": td_texts[3] if len(td_texts) > 3 else ""
                        })

            if allottee_data:
                pd.DataFrame(allottee_data).to_csv(f"{quarter_path}/Allottee_Details.csv", index=False)
                print("    ✅ Saved Allottee_Details.csv")
                
        except Exception as e:
            pass # Silent fail if Allottee Details aren't on the page

print("\n🎉 Finished processing all files!")
driver.quit()

🧹 Setting up a clean environment...
⬇️ Fetching the correct driver...
🚀 Booting Chrome...

--- Starting Data Extraction Phase ---

📁 Processing Sub-Project: DREAM COLONY

  -> Opening Quarter: October to December, 2025
  -> Extracting QPR Forms (R1, R2, R3) via unified parser...
    ✅ Saved Form_R1.xlsx
    ✅ Saved Form_R2.xlsx
    ✅ Saved Form_R3.xlsx
    ✅ Saved Allottee_Details.csv

  -> Opening Quarter: July to September, 2025
  -> Extracting QPR Forms (R1, R2, R3) via unified parser...
    ✅ Saved Form_R1.xlsx
    ✅ Saved Form_R2.xlsx
    ✅ Saved Form_R3.xlsx
    ✅ Saved Allottee_Details.csv

  -> Opening Quarter: April to June, 2025
  -> Extracting QPR Forms (R1, R2, R3) via unified parser...
    ✅ Saved Form_R1.xlsx
    ✅ Saved Form_R2.xlsx
    ✅ Saved Form_R3.xlsx
    ✅ Saved Allottee_Details.csv

  -> Opening Quarter: January to March, 2025
  -> Extracting QPR Forms (R1, R2, R3) via unified parser...
    ✅ Saved Form_R1.xlsx
    ✅ Saved Form_R2.xlsx
    ✅ Saved Form_R3.xlsx
  